# Least Count — Trained Bot

Self-contained notebook to train a neural-net policy that plays Least Count better than the
heuristic Smart bot. Run cells top to bottom.

**Runtime:** Python 3, T4 GPU.

**Pipeline:**
1. Setup + GPU check.
2. Engine + tests (must all pass).
3. Smart bot + encoder + network.
4. Behavior cloning warm-start (~5 min).
5. PPO self-play training with league play (~6–10 hours).
6. Evaluation vs Smart.
7. Export weights as JSON for the browser.

Checkpoints save every 30 min; the best-vs-Smart model is saved separately.


## Cell 1: Setup + GPU check

In [ ]:
import torch, numpy, sys
print('python', sys.version)
print('torch', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('using device:', device)


## Cell 2: Engine (Python port of the JS Game class)

Bit-for-bit equivalent to `index.html`.

In [ ]:
"""
Python port of the Least Count game engine.
Mirrors the JS engine in index.html exactly. All 220 JS tests are reproduced in test_engine.py
and must pass before any training begins.
"""
import random
import copy
from typing import List, Optional, Dict, Any

RANKS = ['A','2','3','4','5','6','7','8','9','10','J','Q','K']
SUITS = ['S','H','D','C']
DECLARE_MAX = 5
SKIP_RANK = 'J'
WAR_RANK = '7'
WAR_PER_CARD = 2
BULK_NO_DRAW = 3


def rank_value(r: str) -> int:
    if r == 'JOKER': return 0
    if r == 'A': return 1
    if r in ('J','Q','K'): return 10
    return int(r)


def make_deck() -> List[Dict[str, Any]]:
    c = []
    for k in range(2):
        for s in SUITS:
            for r in RANKS:
                c.append({'id': r + s + ('a' if k == 0 else 'b'), 'rank': r, 'suit': s})
    c.append({'id': 'JOKERa', 'rank': 'JOKER', 'suit': None})
    c.append({'id': 'JOKERb', 'rank': 'JOKER', 'suit': None})
    return c


def shuffle(a: List, rng: Optional[random.Random] = None) -> List:
    rng = rng or random
    for i in range(len(a) - 1, 0, -1):
        j = rng.randint(0, i)
        a[i], a[j] = a[j], a[i]
    return a


class Game:
    def __init__(self, target: int = 100, penalty: int = 40,
                 names=None, rng: Optional[random.Random] = None):
        self.rng = rng or random.Random()
        self.target = target
        self.penalty = penalty
        self.names = names or ['Player 1', 'Player 2']
        self.scores = [0, 0]
        self.round = 0
        self.winner: Optional[int] = None
        self.phase = 'lobby'
        # Will be set in start_round
        self.hands: List[List[Dict]] = [[], []]
        self.wild_indicator: Optional[Dict] = None
        self.wild_rank: str = 'A'
        self.floor: List[Dict] = []
        self.deck: List[Dict] = []
        self.graveyard: List[Dict] = []
        self.pending_discard: Optional[List[Dict]] = None
        self.pending_jacks = 0
        self.pending_sevens = 0
        self.pending_penalty = 0
        self.war_starter: Optional[int] = None
        self.turn = 0
        self.starter = 0
        self.skip_eligible = False
        self.reveal = False
        self.round_result: Optional[Dict] = None
        self.last_action = ''
        self.start_round(0)

    def start_round(self, starter: int) -> None:
        self.round += 1
        deck = shuffle(make_deck(), self.rng)
        self.hands = [[], []]
        for _ in range(7):
            self.hands[0].append(deck.pop())
            self.hands[1].append(deck.pop())
        # Wild indicator can't be 7, J, or Joker — push them to bottom and retry
        skipped = []
        self.wild_indicator = deck.pop()
        while self.wild_indicator and self.wild_indicator['rank'] in (WAR_RANK, SKIP_RANK, 'JOKER'):
            skipped.append(self.wild_indicator)
            self.wild_indicator = deck.pop()
        for s in skipped:
            deck.insert(0, s)
        self.wild_rank = self.wild_indicator['rank']
        self.floor = [deck.pop()]
        self.deck = deck
        self.graveyard = []
        self.pending_discard = None
        self.pending_jacks = 0
        self.pending_sevens = 0
        self.war_starter = None
        self.turn = starter
        self.starter = starter
        self.skip_eligible = False
        self.reveal = False
        self.round_result = None
        if self.floor[0]['rank'] == WAR_RANK:
            self.pending_penalty = WAR_PER_CARD
            self.phase = 'war'
            self.last_action = f"Round {self.round} · open pile is a 7 — {self.names[starter]} must counter or pick {self.pending_penalty}."
        elif self.floor[0]['rank'] == SKIP_RANK:
            self.pending_penalty = 0
            self.phase = 'discard'
            self.turn = starter ^ 1
            self.last_action = f"Round {self.round} · open pile is a Jack — {self.names[starter]} skipped, {self.names[starter^1]} starts."
        else:
            self.pending_penalty = 0
            self.phase = 'discard'
            self.last_action = f"Round {self.round} · {self.names[starter]} starts."

    def card_points(self, c: Dict) -> int:
        if c['rank'] == 'JOKER': return 0
        if c['rank'] == self.wild_rank: return 0
        return rank_value(c['rank'])

    def hand_points(self, h: List[Dict]) -> int:
        return sum(self.card_points(c) for c in h)

    def floor_rank(self) -> Optional[str]:
        return self.floor[0]['rank'] if self.floor else None

    def is_valid_set(self, cards: List[Dict]) -> bool:
        if not cards: return False
        r = cards[0]['rank']
        return all(c['rank'] == r for c in cards)

    def can_declare(self, p: int) -> bool:
        return (self.phase == 'discard' and self.turn == p and
                self.pending_penalty == 0 and
                self.hand_points(self.hands[p]) <= DECLARE_MAX)

    def declare(self, p: int) -> None:
        if self.phase == 'war':
            raise ValueError(f"Answer the 7-penalty first — throw a 7 or pick up {self.pending_penalty}.")
        if not self.can_declare(p):
            raise ValueError(f"You can only declare on your turn with a hand of {DECLARE_MAX} or less.")
        self._resolve_show(p)

    def discard(self, p: int, ids: List[str]) -> Dict:
        if self.turn != p: raise ValueError("Not your move.")
        if self.phase not in ('discard', 'war'): raise ValueError("Not your move.")
        hand = self.hands[p]
        if not ids: raise ValueError("Pick at least one card.")
        # Map ids -> card refs from hand (preserving order, no duplicates)
        chosen = []
        for cid in ids:
            found = next((c for c in hand if c['id'] == cid), None)
            if found is not None and found not in chosen:
                chosen.append(found)
        if len(chosen) != len(ids): raise ValueError("Card not in hand.")
        if not self.is_valid_set(chosen): raise ValueError("Same-rank only.")
        is_seven = chosen[0]['rank'] == WAR_RANK
        is_jack = chosen[0]['rank'] == SKIP_RANK
        if self.phase == 'war' and not is_seven:
            # End war: throw + pick penalty
            jacks_thrown = len(chosen) if is_jack else 0
            self.hands[p] = [c for c in hand if c['id'] not in ids]
            self.graveyard.extend(self.floor)
            self.floor = chosen
            self.pending_discard = None
            self.pending_jacks = 0
            self.pending_sevens = 0
            self.skip_eligible = False
            pick_count = self.pending_penalty
            picked = 0
            for _ in range(pick_count):
                if not self.deck: self._reshuffle()
                if not self.deck: break
                self.hands[p].append(self.deck.pop())
                picked += 1
            self.pending_penalty = 0
            self.war_starter = None
            self.phase = 'discard'
            self.turn = p ^ ((jacks_thrown + 1) % 2)
            self.last_action = f"{self.names[p]} threw and picked {picked}."
            self._auto_win_if_empty()
            return {'warEnded': True, 'picked': picked, 'jacksThrown': jacks_thrown}
        self.hands[p] = [c for c in hand if c['id'] not in ids]
        self.pending_discard = chosen
        self.pending_jacks = len(chosen) if is_jack else 0
        self.pending_sevens = len(chosen) if is_seven else 0
        matches_pile = self.floor_rank() is not None and chosen[0]['rank'] == self.floor_rank()
        self.skip_eligible = matches_pile
        bulk = len(chosen) >= BULK_NO_DRAW
        if bulk or matches_pile:
            self.last_action = f"{self.names[p]} threw (no draw)."
            self._finish_turn()
            return {'skippedDraw': True, 'war': self.phase == 'war', 'matchedPile': matches_pile}
        self.phase = 'draw'
        self.last_action = f"{self.names[p]} threw."
        return {'skipEligible': False, 'pendingSevens': self.pending_sevens}

    def draw(self, p: int, source: str, card_id: Optional[str] = None) -> None:
        if self.phase != 'draw' or self.turn != p:
            raise ValueError("Not your draw.")
        if source == 'deck':
            if not self.deck: self._reshuffle()
            if not self.deck: raise ValueError("No cards left to draw.")
            c = self.deck.pop()
            self.hands[p].append(c)
            self.last_action = f"{self.names[p]} drew from deck."
        elif source == 'floor':
            i = next((idx for idx, c in enumerate(self.floor) if c['id'] == card_id), -1)
            if i == -1: raise ValueError("That card isn't on the pile.")
            c = self.floor[i]
            if c['rank'] == WAR_RANK: raise ValueError("Can't pick a 7 off the pile.")
            if c['rank'] == SKIP_RANK: raise ValueError("Can't pick a Jack off the pile.")
            self.floor.pop(i)
            self.hands[p].append(c)
            self.last_action = f"{self.names[p]} took from pile."
        else:
            raise ValueError("bad source")
        self._finish_turn()

    def skip_draw(self, p: int) -> None:
        if self.phase != 'draw' or self.turn != p:
            raise ValueError("Not your draw.")
        if not self.skip_eligible:
            raise ValueError("Can only skip draw if your throw matched the pile.")
        if len(self.hands[p]) < 1:
            raise ValueError("You'd have no cards.")
        self.last_action = f"{self.names[p]} skipped draw."
        self._finish_turn()

    def _finish_turn(self) -> None:
        self.graveyard.extend(self.floor)
        self.floor = self.pending_discard or []
        self.pending_discard = None
        self.skip_eligible = False
        thrower = self.turn
        sevens = self.pending_sevens
        self.pending_sevens = 0
        if sevens > 0:
            if self.war_starter is None:
                self.war_starter = thrower
            self.pending_penalty += WAR_PER_CARD * sevens
            self.pending_jacks = 0
            self.turn = thrower ^ 1
            self.phase = 'war'
            self.last_action += f" — {self.names[thrower^1]} must counter or pick {self.pending_penalty}."
            self._auto_win_if_empty()
            return
        j = self.pending_jacks
        self.pending_jacks = 0
        self.turn = thrower ^ ((j + 1) % 2)
        self.phase = 'discard'
        if j > 0:
            self.last_action += f" — {j} Jack(s)!"
        self._auto_win_if_empty()

    def _auto_win_if_empty(self) -> None:
        if self.phase in ('roundover', 'gameover'): return
        if len(self.hands[self.turn]) != 0: return
        p = self.turn
        # If a 7-war landed on empty-handed player, auto-pick penalty
        if self.phase == 'war' and self.pending_penalty > 0:
            picked = 0
            for _ in range(self.pending_penalty):
                if not self.deck: self._reshuffle()
                if not self.deck: break
                self.hands[p].append(self.deck.pop())
                picked += 1
            self.pending_penalty = 0
            self.war_starter = None
            self.pending_sevens = 0
            self.phase = 'discard'
            self.turn = p ^ 1
            self.last_action = f"{self.names[p]} auto-picked {picked} from 7-penalty."
            return
        # Auto-win
        self.pending_penalty = 0
        self.pending_sevens = 0
        self.pending_jacks = 0
        self.war_starter = None
        self.pending_discard = None
        self._resolve_show(p)
        self.last_action = f"{self.names[p]} ran out of cards — auto-win!"

    def _reshuffle(self) -> None:
        if not self.graveyard: return
        self.deck = shuffle(self.graveyard, self.rng)
        self.graveyard = []

    def _resolve_show(self, declarer: int) -> None:
        t = [self.hand_points(self.hands[0]), self.hand_points(self.hands[1])]
        o = declarer ^ 1
        correct = t[declarer] <= t[o]
        d = [0, 0]
        cap = self.penalty
        if correct:
            d[declarer] = 0
            d[o] = min(t[o], cap)
        else:
            d[declarer] = cap
            d[o] = 0
        self.scores[0] += d[0]
        self.scores[1] += d[1]
        self.round_result = {
            'declarer': declarer, 'totals': t, 'correct': correct, 'deltas': d
        }
        self.reveal = True
        self.phase = 'roundover'
        self.last_action = f"{self.names[declarer]} declared with {t[declarer]} — {'correct' if correct else 'WRONG'}."
        if self.scores[0] >= self.target or self.scores[1] >= self.target:
            self.winner = 0 if self.scores[0] < self.scores[1] else 1
            self.phase = 'gameover'

    def next_round(self) -> None:
        if self.phase == 'gameover': return
        self.start_round(self.starter ^ 1)

    def view_for(self, p: int) -> Dict[str, Any]:
        opp = p ^ 1
        return {
            'you': p,
            'names': list(self.names),
            'target': self.target,
            'penalty': self.penalty,
            'round': self.round,
            'scores': list(self.scores),
            'yourHand': [{'id': c['id'], 'rank': c['rank'], 'suit': c['suit'],
                          'pts': self.card_points(c)} for c in self.hands[p]],
            'yourPoints': self.hand_points(self.hands[p]),
            'oppHandCount': len(self.hands[opp]),
            'oppHand': ([{'id': c['id'], 'rank': c['rank'], 'suit': c['suit'],
                          'pts': self.card_points(c)} for c in self.hands[opp]]
                        if self.reveal else None),
            'oppPoints': self.hand_points(self.hands[opp]) if self.reveal else None,
            'floor': [{'id': c['id'], 'rank': c['rank'], 'suit': c['suit'],
                       'pts': self.card_points(c)} for c in self.floor],
            'pendingDiscard': ([{'id': c['id'], 'rank': c['rank'], 'suit': c['suit'],
                                 'pts': self.card_points(c)} for c in self.pending_discard]
                               if self.pending_discard else None),
            'deckCount': len(self.deck),
            'wildRank': self.wild_rank,
            'wildIndicator': self.wild_indicator,
            'turn': self.turn,
            'yourTurn': self.turn == p,
            'phase': self.phase,
            'skipEligible': self.skip_eligible,
            'pendingPenalty': self.pending_penalty,
            'pendingSevens': self.pending_sevens,
            'canDeclare': self.can_declare(p),
            'reveal': self.reveal,
            'roundResult': self.round_result,
            'winner': self.winner,
            'lastAction': self.last_action,
            'declareMax': DECLARE_MAX,
        }


## Cell 3: Engine tests (must all pass)

If this fails, STOP — engine bug would corrupt training.

In [ ]:
"""
Core test suite for the Python engine port. Mirrors the critical scenarios from /tmp/lc_test.js.
"""
import random
# engine symbols already in scope

pass_n = 0
fail_n = 0


def ok(name, cond, info=''):
    global pass_n, fail_n
    if cond:
        pass_n += 1
        print(f'  ok   {name}')
    else:
        fail_n += 1
        print(f'  FAIL {name} {info}')


def H(s):
    """Parse card string like '5H', '10C', 'JK1', 'JK2'."""
    if s == 'JK1': return {'id': 'JOKERa', 'rank': 'JOKER', 'suit': None}
    if s == 'JK2': return {'id': 'JOKERb', 'rank': 'JOKER', 'suit': None}
    # Optional trailing a/b for 2-deck disambiguation
    import re
    m = re.match(r'^(10|[A2-9JQK])([SHDC])(\d?)$', s)
    if not m: raise ValueError(f'bad card {s}')
    suf = m.group(3) or ''
    return {'id': m.group(1) + m.group(2) + suf, 'rank': m.group(1), 'suit': m.group(2)}


def make_game(target=100, seed=None):
    rng = random.Random(seed) if seed is not None else None
    return Game(target=target, names=['A','B'], rng=rng)


def reset(g, turn=0):
    g.turn = turn
    g.phase = 'discard'
    g.pending_penalty = 0
    g.pending_discard = None
    g.pending_jacks = 0
    g.pending_sevens = 0
    g.skip_eligible = False
    g.war_starter = None
    g.graveyard = []


def set_hands(g, h0, h1):
    g.hands[0] = [H(s) for s in h0]
    g.hands[1] = [H(s) for s in h1]


def set_floor(g, cards):
    g.floor = [H(s) for s in cards]


def set_deck(g, top):
    """top[0] is the next card to be popped."""
    g.deck = [H(s) for s in reversed(top)]


def set_wild(g, rank):
    g.wild_indicator = {'id': rank + 'S', 'rank': rank, 'suit': 'S'}
    g.wild_rank = rank


# ==================== TESTS ====================

print('=== ENGINE PORT TESTS ===\n')

print('--- 1: makeDeck has 106 cards, all unique, 8 sevens, 2 jokers ---')
d = make_deck()
ok('106 cards', len(d) == 106, f'count={len(d)}')
ids = set(c['id'] for c in d)
ok('all ids unique', len(ids) == 106, f'unique={len(ids)}')
sevens = [c for c in d if c['rank'] == '7']
ok('8 sevens', len(sevens) == 8)
jokers = [c for c in d if c['rank'] == 'JOKER']
ok('2 jokers', len(jokers) == 2)

print('--- 2: pile-match auto-skips draw (any rank) ---')
g = make_game()
set_hands(g, ['QH','3S'], ['2H','3D','4C','5D','6D','8H','10H'])
set_floor(g, ['QC']); set_wild(g, 'K'); reset(g, 0); set_deck(g, ['AC'])
g.discard(0, ['QH'])
ok('phase=discard (auto-skip)', g.phase == 'discard')
ok('no draw', len(g.hands[0]) == 1)
ok('turn=1', g.turn == 1)

print('--- 3: pile-match for 7 on 7 -> war ---')
g = make_game()
set_hands(g, ['7H','5S'], ['2H','3D','4C','5D','6D','8H','10H'])
set_floor(g, ['7C']); set_wild(g, 'K'); reset(g, 0); set_deck(g, ['AC','AD'])
g.discard(0, ['7H'])
ok('phase=war', g.phase == 'war')
ok('penalty=2', g.pending_penalty == 2)
ok('no draw', len(g.hands[0]) == 1)
ok('turn=1', g.turn == 1)

print('--- 4: pile-match for J on J -> play again ---')
g = make_game()
set_hands(g, ['JH','5S'], ['2H','3D','4C','5D','6D','8H','10H'])
set_floor(g, ['JC']); set_wild(g, 'K'); reset(g, 0); set_deck(g, ['AC'])
g.discard(0, ['JH'])
ok('phase=discard', g.phase == 'discard')
ok('turn=0 (1 jack, play again)', g.turn == 0)
ok('no draw', len(g.hands[0]) == 1)

print('--- 5: single 7 on non-7 pile: draw then war ---')
g = make_game()
set_hands(g, ['7H','5S'], ['2H','3D','4C','5D','6D','8H','10H'])
set_floor(g, ['10S']); set_wild(g, 'K'); reset(g, 0); set_deck(g, ['AC','AD'])
g.discard(0, ['7H'])
ok('phase=draw', g.phase == 'draw')
ok('penalty not yet applied', g.pending_penalty == 0)
g.draw(0, 'deck')
ok('phase=war after draw', g.phase == 'war')
ok('penalty=2', g.pending_penalty == 2)
ok('A hand=2', len(g.hands[0]) == 2)
ok('turn=1', g.turn == 1)

print('--- 6: war non-7 throw -> shed + pick + turn ends ---')
g = make_game()
set_hands(g, ['7H','5S'], ['3D','4C','5D','6D','8H','10H','KS'])
set_floor(g, ['10S']); set_wild(g, 'K'); reset(g, 0); set_deck(g, ['AC','AD','AS','AH'])
g.discard(0, ['7H']); g.draw(0, 'deck')
b_before = len(g.hands[1])
g.discard(1, ['3D'])
ok('war ended', g.phase == 'discard')
ok('net +1', len(g.hands[1]) == b_before - 1 + 2)
ok('penalty cleared', g.pending_penalty == 0)
ok('turn=0', g.turn == 0)

print('--- 7: 3 sevens trigger war immediately (no draw) ---')
g = make_game()
set_hands(g, ['7H','7C','7D','5S'], ['2H','3D','4C','5D','6D','8H','10H'])
set_floor(g, ['10S']); set_wild(g, 'K'); reset(g, 0); set_deck(g, ['AC','AD'])
g.discard(0, ['7H','7C','7D'])
ok('phase=war', g.phase == 'war')
ok('penalty=6', g.pending_penalty == 6)
ok('hand=1', len(g.hands[0]) == 1)

print('--- 8: 7s and Js cannot be picked from open pile ---')
g = make_game()
set_hands(g, ['5H'], ['2H'])
set_floor(g, ['7H','JS','5C']); set_wild(g, 'K'); reset(g, 0)
g.phase = 'draw'; g.pending_discard = None
set_deck(g, ['AC'])
t1 = False
try: g.draw(0, 'floor', '7H')
except ValueError: t1 = True
ok('7 rejected', t1)
t2 = False
try: g.draw(0, 'floor', 'JS')
except ValueError: t2 = True
ok('J rejected', t2)
g.draw(0, 'floor', '5C')
ok('5 allowed', any(c['id'] == '5C' for c in g.hands[0]))

print('--- 9: wild rank never 7, J, or Joker (10000 deals) ---')
bad = 0
for _ in range(10000):
    g = Game(target=100, names=['A','B'])
    if g.wild_indicator['rank'] in ('7','J','JOKER'):
        bad += 1
ok('no 7/J/Joker wild', bad == 0, f'bad={bad}')

print('--- 10: declare correct - opp count capped at 40 ---')
g = make_game()
set_hands(g, ['KH','AS'], ['10H','10S','10D','10C','9H'])  # K=0 (wild), A=1 → 1 pt
set_floor(g, ['QS']); set_wild(g, 'K'); reset(g, 0)
g.declare(0)
ok('correct=True', g.round_result['correct'] == True)
ok('A=0', g.round_result['deltas'][0] == 0)
ok('B capped at 40', g.round_result['deltas'][1] == 40)

print('--- 11: declare tie -> declarer wins ---')
g = make_game()
set_hands(g, ['AS','3D'], ['AH','3C'])  # both 4 pts
set_floor(g, ['QS']); set_wild(g, 'K'); reset(g, 0)
g.declare(0)
ok('correct (tie to declarer)', g.round_result['correct'] == True)
ok('A=0', g.round_result['deltas'][0] == 0)
ok('B=4', g.round_result['deltas'][1] == 4)

print('--- 12: declare wrong (opp strictly beats) ---')
g = make_game()
set_hands(g, ['AS','3D'], ['AH'])  # A=4, B=1
set_floor(g, ['QS']); set_wild(g, 'K'); reset(g, 0)
g.declare(0)
ok('wrong', g.round_result['correct'] == False)
ok('A=+40', g.round_result['deltas'][0] == 40)
ok('B=0', g.round_result['deltas'][1] == 0)

print('--- 13: declare blocked in draw/war ---')
g = make_game()
set_hands(g, ['7H','AS'], ['AS','AH','2S','3D'])
set_floor(g, ['10S']); set_wild(g, '2'); reset(g, 0); set_deck(g, ['AC','AD'])
g.discard(0, ['7H'])
t1 = False
try: g.declare(0)
except ValueError: t1 = True
ok('declare blocked in draw', t1)
g.draw(0, 'deck')
t2 = False
try: g.declare(1)
except ValueError: t2 = True
ok('declare blocked in war', t2)

print('--- 14: open-card-7 at round start triggers war for starter ---')
found = 0
validated = 0
for _ in range(2000):
    g = Game(target=100, names=['A','B'])
    if g.floor[0]['rank'] == '7':
        found += 1
        if g.phase == 'war' and g.pending_penalty == 2 and g.turn == g.starter:
            validated += 1
ok(f'7-start found ({found} of 2000)', found > 50)
ok('every 7-start is war for starter', validated == found, f'{validated}/{found}')

print('--- 15: open-card-J at round start skips starter ---')
found = 0
validated = 0
for _ in range(3000):
    g = Game(target=100, names=['A','B'])
    if g.floor[0]['rank'] == 'J':
        found += 1
        if g.phase == 'discard' and g.turn == (g.starter ^ 1) and g.pending_penalty == 0:
            validated += 1
ok(f'J-start found ({found} of 3000)', found > 50)
ok('every J-start has turn flipped', validated == found, f'{validated}/{found}')

print('--- 16: 3+ throw empties hand -> auto-win on return ---')
g = make_game()
set_hands(g, ['5H','5D','5C'], ['2H','3D','4C','6D','8H','10H','KS'])
set_floor(g, ['QS']); set_wild(g, 'K'); reset(g, 0); set_deck(g, ['AC','AD','AS','AH'])
g.discard(0, ['5H','5D','5C'])
ok('A hand=0', len(g.hands[0]) == 0)
ok('turn=1', g.turn == 1)
g.discard(1, ['2H']); g.draw(1, 'deck')
ok('A auto-wins', g.phase in ('roundover','gameover'))
ok('declarer=A', g.round_result['declarer'] == 0)

print('--- 17: empty hand + opp throws 7 -> auto-pick penalty, game continues ---')
g = make_game()
set_hands(g, ['5H','5D','5C'], ['7H','3D','4C','6D','8H','10H','KS'])
set_floor(g, ['QS']); set_wild(g, 'K'); reset(g, 0); set_deck(g, ['AC','AD','AS','AH','2C','2D','2S'])
g.discard(0, ['5H','5D','5C'])
g.discard(1, ['7H']); g.draw(1, 'deck')
ok('A picked 2, game continues', len(g.hands[0]) == 2 and g.phase == 'discard')
ok('penalty cleared', g.pending_penalty == 0)
ok('turn=B', g.turn == 1)

print('--- 18: J ends 7-war -> picks penalty AND J skip applies ---')
g = make_game()
set_hands(g, ['7H','9C'], ['JC','3D','5D','6D','8H','10H','KS'])
set_floor(g, ['10S']); set_wild(g, 'K'); reset(g, 0); set_deck(g, ['AC','AD','AS','AH'])
g.discard(0, ['7H']); g.draw(0, 'deck')
b_before = len(g.hands[1])
g.discard(1, ['JC'])
ok('B threw J, picked 2 (net +1)', len(g.hands[1]) == b_before - 1 + 2)
ok('turn=1 (B plays again — odd jacks)', g.turn == 1)
ok('floor=J', g.floor[0]['id'] == 'JC')

print('--- 19: 2 sevens stacked, opp counters with 1 seven, war continues ---')
g = make_game()
set_hands(g, ['7H','7C','9C'], ['7D','3D','4C','5D','6D','8H','10H'])
set_floor(g, ['10S']); set_wild(g, 'K'); reset(g, 0); set_deck(g, ['AC','AD','AS','AH','2C','2D'])
g.discard(0, ['7H','7C']); g.draw(0, 'deck')
ok('penalty=4 turn=B', g.pending_penalty == 4 and g.turn == 1)
g.discard(1, ['7D'])  # matches pile (7s), auto-skip + war bounces
ok('B counter, penalty=6, turn=A', g.pending_penalty == 6 and g.turn == 0 and g.phase == 'war')

print('--- 20: serialization-free round-trip via __dict__ ---')
import copy
g = make_game()
set_hands(g, ['7H','5S'], ['3D','4C','5D','6D','8H','10H','KS'])
set_floor(g, ['10S']); set_wild(g, 'K'); reset(g, 0); set_deck(g, ['AC','AD'])
g.discard(0, ['7H'])
snap = copy.deepcopy(g.__dict__)
# Continue
g2 = make_game()
g2.__dict__.update(copy.deepcopy(snap))
ok('phase preserved', g2.phase == 'draw')
ok('pendingSevens preserved', g2.pending_sevens == 1)
g2.draw(0, 'deck')
ok('war resumes', g2.phase == 'war' and g2.pending_penalty == 2)

print('--- 21: reshuffle when deck empties ---')
g = make_game()
set_hands(g, ['5H','6S'], ['2H','3D'])
set_floor(g, ['10S']); set_wild(g, 'K'); reset(g, 0)
g.deck = []
g.graveyard = [H('AC'), H('AD'), H('AS'), H('AH'), H('2C'), H('2D')]
g.discard(0, ['5H']); g.draw(0, 'deck')
ok('A got reshuffled card', len(g.hands[0]) == 2)

print('--- 22: PROPERTY — 5000 random games complete without crashing ---')
crashed = 0
rounds_completed = 0
for trial in range(5000):
    g = Game(target=50, names=['A','B'], rng=random.Random(trial))
    steps = 0
    while g.phase != 'gameover' and steps < 2000:
        p = g.turn
        try:
            if g.phase == 'discard':
                # Random valid action: declare if eligible else throw random card(s)
                if g.can_declare(p) and random.random() < 0.05:
                    g.declare(p)
                else:
                    h = g.hands[p]
                    if not h: break  # shouldn't happen but defensive
                    # Randomly throw 1 card
                    c = random.choice(h)
                    g.discard(p, [c['id']])
            elif g.phase == 'war':
                h = g.hands[p]
                if not h: break
                # Random valid action
                sevens = [c for c in h if c['rank'] == '7']
                if sevens:
                    g.discard(p, [sevens[0]['id']])
                else:
                    c = random.choice(h)
                    g.discard(p, [c['id']])
            elif g.phase == 'draw':
                if random.random() < 0.3:
                    # Try to draw from pile (skip 7/J)
                    drawable = [c for c in g.floor if c['rank'] not in ('7','J')]
                    if drawable:
                        g.draw(p, 'floor', drawable[0]['id'])
                    else:
                        g.draw(p, 'deck')
                else:
                    g.draw(p, 'deck')
            elif g.phase == 'roundover':
                g.next_round()
                rounds_completed += 1
        except Exception as e:
            crashed += 1
            print(f'  CRASH trial {trial} step {steps}: {e}')
            break
        steps += 1
ok(f'5000 games no crashes ({rounds_completed} rounds total)', crashed == 0)

print(f'\nResults: {pass_n} pass, {fail_n} fail')
assert fail_n == 0, f'{fail_n} test(s) failed — stop and fix before training'


## Cell 4: Smart bot (baseline + behavior cloning teacher)

In [ ]:
"""
Python port of botSmart and botEasy. Used as:
- Baseline opponent
- Behavior-cloning teacher
- League opponent during PPO training
"""
import random
from typing import List, Dict, Any, Optional
# Game already in scope


def generate_throws(hand: List[Dict]) -> List[List[Dict]]:
    """Each single card + every same-rank set of size 2..n."""
    moves = []
    for c in hand:
        moves.append([c])
    by_rank: Dict[str, List[Dict]] = {}
    for c in hand:
        by_rank.setdefault(c['rank'], []).append(c)
    for r, cs in by_rank.items():
        if len(cs) >= 2:
            for k in range(2, len(cs) + 1):
                moves.append(cs[:k])
    return moves


def bot_easy(g: Game, p: int, rng: Optional[random.Random] = None) -> Dict[str, Any]:
    """Random valid move; sometimes declares foolishly."""
    rng = rng or random
    view = g.view_for(p)
    if view['phase'] == 'war':
        hand = g.hands[p]
        non_sevens = [c for c in hand if c['rank'] != '7']
        if non_sevens:
            return {'t': 'discard', 'cardIds': [rng.choice(non_sevens)['id']]}
        return {'t': 'discard', 'cardIds': [hand[0]['id']]}
    if view['phase'] == 'discard':
        if g.hand_points(g.hands[p]) <= 5 and rng.random() < 0.35:
            return {'t': 'declare'}
        moves = generate_throws(g.hands[p])
        m = rng.choice(moves)
        return {'t': 'discard', 'cardIds': [c['id'] for c in m]}
    if view['phase'] == 'draw':
        pickable = [c for c in view['floor'] if c['rank'] not in ('7','J')]
        if pickable and rng.random() < 0.4:
            return {'t': 'draw', 'source': 'floor', 'cardId': rng.choice(pickable)['id']}
        return {'t': 'draw', 'source': 'deck'}
    return {'t': 'draw', 'source': 'deck'}


def score_throw_smart(g: Game, p: int, cards: List[Dict], view: Dict) -> float:
    hand_before = g.hands[p]
    chosen_ids = {c['id'] for c in cards}
    hand_after = [c for c in hand_before if c['id'] not in chosen_ids]
    pts_after = sum(g.card_points(c) for c in hand_after)
    pts_removed = sum(g.card_points(c) for c in cards)
    is_bulk = len(cards) >= 3
    is_seven = cards[0]['rank'] == '7'
    is_jack = cards[0]['rank'] == 'J'
    matches_pile = (len(view['floor']) > 0 and view['floor'][0]['rank'] == cards[0]['rank'])
    skips_draw = is_bulk or matches_pile
    score = -pts_after + pts_removed * 0.4
    if skips_draw: score += 8
    if len(hand_after) == 0: score += 120
    elif pts_after <= 5: score += 30
    if is_seven:
        if view['oppHandCount'] <= 4: score -= 10
        else: score += 4
        if len(hand_after) == 0: score += 20
    if is_jack:
        if pts_after <= 8: score += 14
        elif pts_after <= 15: score += 2
        else: score -= 4
        if len(cards) >= 3 and len(cards) % 2 == 1: score += 4
    if len(cards) == 1 and g.card_points(cards[0]) == 0: score -= 6
    if len(cards) == 1 and cards[0]['rank'] == 'A': score -= 3
    return score


def should_declare_smart(g: Game, p: int, view: Dict) -> bool:
    pts = g.hand_points(g.hands[p])
    if pts > 5: return False
    if view['oppHandCount'] >= 3: return True
    return pts == 0


def smart_draw(g: Game, p: int, view: Dict) -> Dict[str, Any]:
    for c in view['floor']:
        if c['rank'] in ('7','J'): continue
        if g.card_points(c) == 0:
            return {'t': 'draw', 'source': 'floor', 'cardId': c['id']}
        if any(x['rank'] == c['rank'] for x in g.hands[p]):
            return {'t': 'draw', 'source': 'floor', 'cardId': c['id']}
    return {'t': 'draw', 'source': 'deck'}


def smart_war(g: Game, p: int) -> Dict[str, Any]:
    hand = g.hands[p]
    sevens = [c for c in hand if c['rank'] == '7']
    if sevens:
        return {'t': 'discard', 'cardIds': [c['id'] for c in sevens]}
    moves = [m for m in generate_throws(hand) if m[0]['rank'] != '7']
    best = None
    best_s = float('-inf')
    for m in moves:
        pts = sum(g.card_points(c) for c in m)
        s = pts + len(m) * 1.5
        if s > best_s:
            best_s = s
            best = m
    return {'t': 'discard', 'cardIds': [c['id'] for c in best]}


def bot_smart(g: Game, p: int) -> Dict[str, Any]:
    view = g.view_for(p)
    if view['phase'] == 'war':
        return smart_war(g, p)
    if view['phase'] == 'discard':
        if should_declare_smart(g, p, view):
            return {'t': 'declare'}
        moves = generate_throws(g.hands[p])
        best = None
        best_s = float('-inf')
        for m in moves:
            s = score_throw_smart(g, p, m, view)
            if s > best_s:
                best_s = s
                best = m
        return {'t': 'discard', 'cardIds': [c['id'] for c in best]}
    if view['phase'] == 'draw':
        return smart_draw(g, p, view)
    return {'t': 'draw', 'source': 'deck'}


def apply_action(g: Game, p: int, a: Dict[str, Any]) -> None:
    t = a['t']
    if t == 'discard': g.discard(p, a['cardIds'])
    elif t == 'draw': g.draw(p, a['source'], a.get('cardId'))
    elif t == 'declare': g.declare(p)


def play_game(bot_a, bot_b, target: int = 50, seed: Optional[int] = None,
              max_steps: int = 5000) -> Optional[int]:
    """Play a full match. bot_a is player 0, bot_b is player 1. Returns winner index or None."""
    rng = random.Random(seed) if seed is not None else None
    g = Game(target=target, names=['A','B'], rng=rng)
    steps = 0
    while g.phase != 'gameover' and steps < max_steps:
        p = g.turn
        try:
            a = bot_a(g, p) if p == 0 else bot_b(g, p)
            apply_action(g, p, a)
        except Exception:
            break
        if g.phase == 'roundover':
            g.next_round()
        steps += 1
    return g.winner


## Cell 5: State encoder + action space

In [ ]:
"""
State encoder + action space for RL training.

State (96 dims) — all info the bot can observe:
- Hand rank counts (14 dims: 13 ranks + Joker), normalized by 8
- Floor top rank one-hot (14 dims)
- Floor size (1) / pickable count (1) / hand-matches-pile binary (1)
- Wild rank one-hot (14 dims)
- Deck count (1) / opp hand count (1)
- Graveyard rank counts (14 dims) — CARD COUNTING
- My score (1) / opp score (1) / target (1) / round (1) / am-I-starter (1)
- Phase one-hot (3 dims: discard/draw/war)
- Pending penalty (1) / pending jacks (1)
- Opp pile picks per rank this round (14 dims)
- Last opp throw rank one-hot (14 dims) / last opp throw count (1)
- My #pairs (1) / #triples (1) / max set size (1)

Action space (45 actions):
- For each of 14 ranks (A,2,3,4,5,6,7,8,9,10,J,Q,K,JOKER):
    throw_1, throw_2, throw_3plus  → 42 actions
- declare (1)
- draw_deck (1)
- draw_pile (1)  [picks pile slot 0; pile cards are homogeneous after a throw]

Legal-action mask per state.
"""
import numpy as np
from collections import Counter
from typing import Optional, Dict, Any, List
# Game already in scope

# Rank list including Joker for action/state indexing
ALL_RANKS = ['A','2','3','4','5','6','7','8','9','10','J','Q','K','JOKER']  # 14
RANK_INDEX = {r: i for i, r in enumerate(ALL_RANKS)}

STATE_DIM = (
    14 +   # hand rank counts
    14 +   # floor top rank one-hot
    1 + 1 + 1 +   # floor_size, pickable_count, hand_matches_pile
    14 +   # wild rank one-hot
    1 + 1 +   # deck_count, opp_hand_count
    14 +   # graveyard rank counts
    1 + 1 + 1 + 1 + 1 +   # my_score, opp_score, target, round_num, am_starter
    3 +    # phase one-hot
    1 + 1 +   # pending_penalty, pending_jacks
    14 +   # opp pile picks per rank (this round)
    14 + 1 +   # last_opp_throw_rank, last_opp_throw_count
    1 + 1 + 1   # my_num_pairs, my_num_triples, my_max_set_size
)
assert STATE_DIM == 103, f'STATE_DIM is {STATE_DIM}'

# Action space layout
N_THROW_RANKS = 14
ACTION_THROW1_BASE = 0           # actions 0..13: throw 1 of rank
ACTION_THROW2_BASE = 14          # actions 14..27: throw pair
ACTION_THROW3_BASE = 28          # actions 28..41: throw 3+ (all of rank)
ACTION_DECLARE = 42
ACTION_DRAW_DECK = 43
ACTION_DRAW_PILE = 44
ACTION_DIM = 45


class Observer:
    """Tracks observable game state across actions for one bot player."""

    def __init__(self, bot_p: int):
        self.bot_p = bot_p
        self.opp_pile_picks: Dict[str, int] = {r: 0 for r in ALL_RANKS}
        self.last_opp_throw_rank: Optional[str] = None
        self.last_opp_throw_count: int = 0
        self.last_round_seen: int = 0

    def maybe_reset(self, g: Game) -> None:
        if g.round != self.last_round_seen:
            self.opp_pile_picks = {r: 0 for r in ALL_RANKS}
            self.last_opp_throw_rank = None
            self.last_opp_throw_count = 0
            self.last_round_seen = g.round

    def record_opp_action(self, p: int, a: Dict, g_before_floor: List[Dict]) -> None:
        if p == self.bot_p: return
        if a.get('t') == 'draw' and a.get('source') == 'floor':
            cid = a.get('cardId')
            for c in g_before_floor:
                if c['id'] == cid:
                    self.opp_pile_picks[c['rank']] = self.opp_pile_picks.get(c['rank'], 0) + 1
                    break
        if a.get('t') == 'discard':
            ids = a.get('cardIds', [])
            # We don't have ranks from id alone, but ids include rank prefix. Parse it.
            rank = None
            for cid in ids:
                if cid.startswith('JOKER'):
                    rank = 'JOKER'
                    break
                # Standard ids: '10' or single char rank + suit + optional a/b
                if cid.startswith('10'):
                    rank = '10'; break
                rank = cid[0]
                break
            self.last_opp_throw_rank = rank
            self.last_opp_throw_count = len(ids)


def encode_state(g: Game, bot_p: int, obs: Observer) -> np.ndarray:
    """Return 102-dim float32 state vector from bot_p's perspective."""
    obs.maybe_reset(g)
    s = np.zeros(STATE_DIM, dtype=np.float32)
    idx = 0

    # Hand rank counts (14)
    hand_counts = Counter(c['rank'] for c in g.hands[bot_p])
    for i, r in enumerate(ALL_RANKS):
        s[idx + i] = hand_counts.get(r, 0) / 8.0
    idx += 14

    # Floor top rank one-hot (14)
    if g.floor:
        ftr = g.floor[0]['rank']
        if ftr in RANK_INDEX:
            s[idx + RANK_INDEX[ftr]] = 1.0
    idx += 14

    # Floor size, pickable count, hand-matches-pile
    s[idx] = len(g.floor) / 8.0; idx += 1
    pickable = sum(1 for c in g.floor if c['rank'] not in ('7','J'))
    s[idx] = pickable / 4.0; idx += 1
    if g.floor:
        ftr = g.floor[0]['rank']
        s[idx] = 1.0 if any(c['rank'] == ftr for c in g.hands[bot_p]) else 0.0
    idx += 1

    # Wild rank one-hot (14)
    if g.wild_rank in RANK_INDEX:
        s[idx + RANK_INDEX[g.wild_rank]] = 1.0
    idx += 14

    # Deck count, opp hand count
    s[idx] = len(g.deck) / 106.0; idx += 1
    s[idx] = len(g.hands[bot_p ^ 1]) / 12.0; idx += 1

    # Graveyard rank counts (14) — card counting
    grave_counts = Counter(c['rank'] for c in g.graveyard)
    for i, r in enumerate(ALL_RANKS):
        s[idx + i] = grave_counts.get(r, 0) / 8.0
    idx += 14

    # My score, opp score, target, round, am-starter
    s[idx] = g.scores[bot_p] / max(g.target, 1); idx += 1
    s[idx] = g.scores[bot_p ^ 1] / max(g.target, 1); idx += 1
    s[idx] = g.target / 200.0; idx += 1
    s[idx] = min(g.round, 20) / 20.0; idx += 1
    s[idx] = 1.0 if g.starter == bot_p else 0.0; idx += 1

    # Phase one-hot (3)
    phase_map = {'discard': 0, 'draw': 1, 'war': 2}
    if g.phase in phase_map:
        s[idx + phase_map[g.phase]] = 1.0
    idx += 3

    # Pending penalty, pending jacks
    s[idx] = g.pending_penalty / 20.0; idx += 1
    s[idx] = g.pending_jacks / 4.0; idx += 1

    # Opp pile picks per rank (14)
    for i, r in enumerate(ALL_RANKS):
        s[idx + i] = obs.opp_pile_picks.get(r, 0) / 4.0
    idx += 14

    # Last opp throw (14 one-hot + count)
    if obs.last_opp_throw_rank and obs.last_opp_throw_rank in RANK_INDEX:
        s[idx + RANK_INDEX[obs.last_opp_throw_rank]] = 1.0
    idx += 14
    s[idx] = min(obs.last_opp_throw_count, 4) / 4.0; idx += 1

    # My num pairs / triples / max set size
    num_pairs = sum(1 for cnt in hand_counts.values() if cnt == 2)
    num_triples = sum(1 for cnt in hand_counts.values() if cnt >= 3)
    max_set = max(hand_counts.values()) if hand_counts else 0
    s[idx] = num_pairs / 6.0; idx += 1
    s[idx] = num_triples / 3.0; idx += 1
    s[idx] = max_set / 8.0; idx += 1

    assert idx == STATE_DIM, f'encoded {idx}, expected {STATE_DIM}'
    return s


def legal_action_mask(g: Game, p: int) -> np.ndarray:
    """Boolean array (ACTION_DIM,) where True = legal."""
    mask = np.zeros(ACTION_DIM, dtype=bool)
    if g.phase == 'discard' or g.phase == 'war':
        hand_counts = Counter(c['rank'] for c in g.hands[p])
        for r, cnt in hand_counts.items():
            if r not in RANK_INDEX: continue
            ri = RANK_INDEX[r]
            # In war, only sevens can be thrown (rule: must counter or pick)
            #   ... unless we treat any non-7 throw in war as "end war + pick penalty"
            #   The engine allows any throw in war (non-7 ends the war).
            # So all rank throws are legal in war.
            if cnt >= 1: mask[ACTION_THROW1_BASE + ri] = True
            if cnt >= 2: mask[ACTION_THROW2_BASE + ri] = True
            if cnt >= 3: mask[ACTION_THROW3_BASE + ri] = True
        if g.can_declare(p):
            mask[ACTION_DECLARE] = True
    if g.phase == 'draw':
        mask[ACTION_DRAW_DECK] = True
        # Pile draw: legal if any non-7-non-J card on floor
        if any(c['rank'] not in ('7','J') for c in g.floor):
            mask[ACTION_DRAW_PILE] = True
    return mask


def action_to_move(g: Game, p: int, action_idx: int) -> Dict[str, Any]:
    """Convert action index → game action dict the engine accepts."""
    if action_idx == ACTION_DECLARE:
        return {'t': 'declare'}
    if action_idx == ACTION_DRAW_DECK:
        return {'t': 'draw', 'source': 'deck'}
    if action_idx == ACTION_DRAW_PILE:
        # Pick the first non-7-non-J card on floor
        for c in g.floor:
            if c['rank'] not in ('7','J'):
                return {'t': 'draw', 'source': 'floor', 'cardId': c['id']}
        raise ValueError('No pickable card on pile')
    # Throw actions
    if action_idx < ACTION_THROW2_BASE:
        size = 1; rank_idx = action_idx - ACTION_THROW1_BASE
    elif action_idx < ACTION_THROW3_BASE:
        size = 2; rank_idx = action_idx - ACTION_THROW2_BASE
    elif action_idx < ACTION_DECLARE:
        size = 3; rank_idx = action_idx - ACTION_THROW3_BASE
    else:
        raise ValueError(f'Unknown action {action_idx}')
    rank = ALL_RANKS[rank_idx]
    matching = [c for c in g.hands[p] if c['rank'] == rank]
    if size == 1:
        return {'t': 'discard', 'cardIds': [matching[0]['id']]}
    if size == 2:
        return {'t': 'discard', 'cardIds': [matching[0]['id'], matching[1]['id']]}
    # 3+: throw all of this rank
    return {'t': 'discard', 'cardIds': [c['id'] for c in matching]}


def smart_action_to_action_idx(g: Game, p: int, smart_action: Dict[str, Any]) -> int:
    """Convert a Smart bot's natural action dict → action_idx for behavior cloning."""
    t = smart_action.get('t')
    if t == 'declare': return ACTION_DECLARE
    if t == 'draw':
        return ACTION_DRAW_DECK if smart_action['source'] == 'deck' else ACTION_DRAW_PILE
    if t == 'discard':
        ids = smart_action['cardIds']
        # Determine rank
        first_id = ids[0]
        if first_id.startswith('JOKER'):
            rank = 'JOKER'
        elif first_id.startswith('10'):
            rank = '10'
        else:
            rank = first_id[0]
        ri = RANK_INDEX[rank]
        n = len(ids)
        if n == 1: return ACTION_THROW1_BASE + ri
        if n == 2: return ACTION_THROW2_BASE + ri
        return ACTION_THROW3_BASE + ri
    raise ValueError(f'Unknown smart action: {smart_action}')


# ===== Self-tests =====


## Cell 6: Policy + value network

In [ ]:
"""
Policy + value network. Asymmetric actor-critic: critic optionally sees opp hand during training.
"""
import torch
import torch.nn as nn
import torch.nn.functional as F
# STATE_DIM/ACTION_DIM in scope


class PolicyValueNet(nn.Module):
    def __init__(self, state_dim: int = STATE_DIM, action_dim: int = ACTION_DIM,
                 hidden_dim: int = 256):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
        )
        self.policy_head = nn.Linear(hidden_dim, action_dim)
        self.value_head = nn.Linear(hidden_dim, 1)

    def forward(self, state):
        h = self.shared(state)
        return self.policy_head(h), self.value_head(h).squeeze(-1)


def masked_policy(logits: torch.Tensor, mask: torch.Tensor) -> torch.distributions.Categorical:
    """Apply legal-action mask: illegal actions get -inf logit, then softmax."""
    masked_logits = logits.masked_fill(~mask, float('-inf'))
    return torch.distributions.Categorical(logits=masked_logits)


def count_params(net: nn.Module) -> int:
    return sum(p.numel() for p in net.parameters())


## Cell 7: Behavior cloning module

In [ ]:
"""
Behavior cloning: collect (state, smart_action) pairs from Smart vs Smart games,
then supervised pre-train the policy to imitate Smart. This is the warm-start for PPO.
"""
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from typing import List, Tuple
# imports in scope
# imports in scope
# imports in scope
# imports in scope


def collect_smart_samples(num_games: int = 1000, seed: int = 0):
    """Run Smart vs Smart games, record (state, action_idx, mask) tuples."""
    states, actions, masks = [], [], []
    rng = random.Random(seed)
    for gi in range(num_games):
        g = Game(target=100, names=['A','B'], rng=random.Random(seed + gi))
        obs = [Observer(0), Observer(1)]
        steps = 0
        while g.phase != 'gameover' and steps < 1500:
            if g.phase == 'roundover':
                g.next_round()
                continue
            p = g.turn
            obs[p].maybe_reset(g)
            state = encode_state(g, p, obs[p])
            mask = legal_action_mask(g, p)
            try:
                smart_a = bot_smart(g, p)
                aidx = smart_action_to_action_idx(g, p, smart_a)
                if not mask[aidx]:
                    steps += 1
                    apply_action(g, p, smart_a)
                    continue
                states.append(state)
                actions.append(aidx)
                masks.append(mask)
                # record opp throw for observer (other player's view)
                opp = p ^ 1
                obs[opp].record_opp_action(p, smart_a, list(g.floor))
                apply_action(g, p, smart_a)
            except Exception:
                break
            steps += 1
    return np.array(states), np.array(actions), np.array(masks)


def train_bc(net: PolicyValueNet, states, actions, masks,
             epochs: int = 5, batch_size: int = 256, lr: float = 1e-3,
             device='cpu'):
    """Supervised cross-entropy on Smart's action choices."""
    net = net.to(device)
    optimizer = optim.Adam(net.parameters(), lr=lr)
    states_t = torch.from_numpy(states).float().to(device)
    actions_t = torch.from_numpy(actions).long().to(device)
    masks_t = torch.from_numpy(masks).bool().to(device)
    N = len(states_t)
    print(f'  BC training: {N} samples, {epochs} epochs, batch={batch_size}')
    for epoch in range(epochs):
        perm = torch.randperm(N)
        total_loss = 0
        correct = 0
        for i in range(0, N, batch_size):
            idx = perm[i:i+batch_size]
            s = states_t[idx]
            a = actions_t[idx]
            m = masks_t[idx]
            logits, _ = net(s)
            # Mask illegal actions in CE: set their logit to large negative
            logits = logits.masked_fill(~m, -1e9)
            loss = nn.functional.cross_entropy(logits, a)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += loss.item() * len(idx)
            preds = logits.argmax(dim=-1)
            correct += (preds == a).sum().item()
        avg_loss = total_loss / N
        acc = correct / N
        print(f'  epoch {epoch+1}: loss={avg_loss:.4f} acc={acc*100:.1f}%')
    return net


## Cell 8: PPO training module

In [ ]:
"""
PPO self-play training with league play.
Single-process; uses one GPU for the policy net. Plays games sequentially using the Python engine.

Key features:
- Bot policy = same network for both seats in self-play; we sample actions and compute reward at round/match end.
- League: 50% latest, 30% earlier checkpoints, 20% Smart heuristic.
- GAE-based advantage with γ=0.99, λ=0.95.
- PPO clip ε=0.2, value coef 0.5, entropy coef 0.01.
- Checkpoint every 30 min wall time. Best-vs-Smart checkpoint separately.
"""
import random
import time
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from typing import List, Optional
# imports in scope
# imports in scope
# imports in scope
# imports in scope


def pick_opponent(league: List[PolicyValueNet], device, smart_fn,
                  rng: random.Random, smart_frac: float = 0.40):
    """Sample an opponent.
    smart_frac fraction of games are vs Smart (anchors policy to known-good baseline).
    Of the remaining: 60% latest checkpoint, 40% earlier checkpoints (when available)."""
    r = rng.random()
    if r < smart_frac or not league:
        return 'smart', smart_fn
    # Of the non-Smart portion, half-half-ish split between latest and past
    r2 = rng.random()
    if r2 < 0.40 and len(league) > 1:
        opp = rng.choice(league[:-1])
        return 'past', opp
    return 'latest', league[-1]


def policy_action(net: PolicyValueNet, state: np.ndarray, mask: np.ndarray,
                  device: torch.device, deterministic: bool = False):
    """Sample action from policy. Returns (action_idx, logprob, value)."""
    s = torch.from_numpy(state).float().unsqueeze(0).to(device)
    m = torch.from_numpy(mask).bool().unsqueeze(0).to(device)
    with torch.no_grad():
        logits, value = net(s)
    dist = masked_policy(logits, m)
    if deterministic:
        # Pick argmax legal
        masked_logits = logits.masked_fill(~m, float('-inf'))
        action_idx = masked_logits.argmax(dim=-1).item()
    else:
        action_idx = dist.sample().item()
    logprob = dist.log_prob(torch.tensor([action_idx], device=device)).item()
    return action_idx, logprob, value.item()


def opp_action(opponent, g, p, device):
    """Get action from any opponent type (Smart fn or PPO net).
    Note: nn.Module is callable too, so we must check isinstance FIRST."""
    if isinstance(opponent, nn.Module):
        obs = Observer(p)
        obs.maybe_reset(g)
        state = encode_state(g, p, obs)
        mask = legal_action_mask(g, p)
        if not mask.any():
            return None
        aidx, _, _ = policy_action(opponent, state, mask, device, deterministic=False)
        return action_to_move(g, p, aidx)
    # Plain Smart bot function (lambda or similar)
    return opponent(g, p)


def play_self_game(net, opponent, device, target=50, max_steps=2000,
                   rng: Optional[random.Random] = None,
                   bot_p: int = 0, gamma: float = 0.99,
                   shape_reward: bool = True):
    """Play one game: net plays as bot_p, opponent plays as 1-bot_p.
    Returns list of (state, mask, action_idx, logprob, value, reward) for the net's actions.

    Reward shaping (when shape_reward=True):
      - small per-step reward = +0.1 * (hand_pts_before - hand_pts_after) — incentivizes dumping
      - round end: -(my_round_delta - opp_round_delta) (capped at ±40 already by engine)
      - match end: ±20 for win/loss (smaller than before so signal is less dominated by tail)
    """
    g = Game(target=target, names=['A','B'], rng=rng)
    obs = Observer(bot_p)
    traj = []
    last_score_diff = 0
    steps = 0
    pts_before_action = g.hand_points(g.hands[bot_p])
    while g.phase != 'gameover' and steps < max_steps:
        if g.phase == 'roundover':
            diff = g.scores[bot_p] - g.scores[bot_p ^ 1]
            round_reward = -(diff - last_score_diff)
            last_score_diff = diff
            if traj:
                traj[-1] = (*traj[-1][:5], traj[-1][5] + round_reward)
            g.next_round()
            obs.maybe_reset(g)
            pts_before_action = g.hand_points(g.hands[bot_p])
            continue
        p = g.turn
        if p == bot_p:
            obs.maybe_reset(g)
            state = encode_state(g, p, obs)
            mask = legal_action_mask(g, p)
            if not mask.any(): break
            aidx, lp, val = policy_action(net, state, mask, device, deterministic=False)
            a = action_to_move(g, p, aidx)
            try:
                apply_action(g, p, a)
            except Exception:
                break
            # Per-step shaping: reward for hand-pts reduction
            step_reward = 0.0
            if shape_reward:
                pts_after = g.hand_points(g.hands[bot_p])
                step_reward = 0.1 * (pts_before_action - pts_after)
                pts_before_action = pts_after
            traj.append((state, mask, aidx, lp, val, step_reward))
        else:
            a = opp_action(opponent, g, p, device)
            if a is None: break
            opp_floor_snapshot = list(g.floor)
            try:
                obs.record_opp_action(p, a, opp_floor_snapshot)
                apply_action(g, p, a)
            except Exception:
                break
            if shape_reward:
                pts_before_action = g.hand_points(g.hands[bot_p])
        steps += 1
    if g.winner is not None:
        # Match reward (smaller; round rewards already cover most signal)
        match_reward = 20.0 if g.winner == bot_p else -20.0
        if traj:
            traj[-1] = (*traj[-1][:5], traj[-1][5] + match_reward)
    return traj, g.winner


def pretrain_value(net, device, num_games: int = 1000, epochs: int = 3,
                   batch_size: int = 256, lr: float = 5e-4):
    """Fit the value head to Monte Carlo returns from Smart-vs-Smart games.
    Runs AFTER BC. Stabilizes PPO by giving the critic a good initialization."""
    from behavior_clone import collect_smart_samples  # already in scope in notebook
    print(f'  Value pretrain: rolling out {num_games} Smart vs Smart games to collect returns...')
    # Replay Smart-vs-Smart games, compute MC returns at each (bot, state).
    samples = []
    rng = random.Random(99)
    for gi in range(num_games):
        g = Game(target=50, names=['A','B'], rng=random.Random(gi + 7777))
        per_player_traj = {0: [], 1: []}
        per_player_last_diff = {0: 0, 1: 0}
        per_player_obs = {0: Observer(0), 1: Observer(1)}
        steps = 0
        while g.phase != 'gameover' and steps < 1500:
            if g.phase == 'roundover':
                for p in (0, 1):
                    diff = g.scores[p] - g.scores[p ^ 1]
                    rr = -(diff - per_player_last_diff[p])
                    per_player_last_diff[p] = diff
                    if per_player_traj[p]:
                        s, _ = per_player_traj[p][-1]
                        per_player_traj[p][-1] = (s, per_player_traj[p][-1][1] + rr)
                g.next_round()
                continue
            p = g.turn
            per_player_obs[p].maybe_reset(g)
            state = encode_state(g, p, per_player_obs[p])
            per_player_traj[p].append((state, 0.0))
            try:
                a = bot_smart(g, p)
                per_player_obs[p ^ 1].record_opp_action(p, a, list(g.floor))
                apply_action(g, p, a)
            except Exception:
                break
            steps += 1
        if g.winner is not None:
            for p in (0, 1):
                mr = 20.0 if g.winner == p else -20.0
                if per_player_traj[p]:
                    s, r = per_player_traj[p][-1]
                    per_player_traj[p][-1] = (s, r + mr)
        # Compute discounted returns
        for p in (0, 1):
            traj = per_player_traj[p]
            G = 0.0
            for i in reversed(range(len(traj))):
                s, r = traj[i]
                G = r + 0.99 * G
                samples.append((s, G))
    print(f'  Collected {len(samples)} (state, return) pairs')
    if not samples: return
    import numpy as np
    states_arr = np.stack([s for s, _ in samples])
    returns_arr = np.array([G for _, G in samples], dtype=np.float32)
    states_t = torch.from_numpy(states_arr).float().to(device)
    returns_t = torch.from_numpy(returns_arr).float().to(device)
    # Train only value head + shared (light)
    optimizer = optim.Adam(net.parameters(), lr=lr)
    N = len(states_t)
    for ep in range(epochs):
        perm = torch.randperm(N)
        total_loss = 0
        for i in range(0, N, batch_size):
            idx = perm[i:i+batch_size]
            _, v = net(states_t[idx])
            loss = ((v - returns_t[idx]) ** 2).mean()
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 0.5)
            optimizer.step()
            total_loss += loss.item() * len(idx)
        print(f'  value epoch {ep+1}: MSE={total_loss/N:.3f}')


def compute_gae(traj, gamma: float = 0.99, lam: float = 0.95):
    """Returns advantages and value targets (returns)."""
    n = len(traj)
    if n == 0: return [], []
    rewards = [t[5] for t in traj]
    values = [t[4] for t in traj]
    advantages = [0.0] * n
    gae = 0.0
    next_value = 0.0  # bootstrapping value past end
    for i in reversed(range(n)):
        delta = rewards[i] + gamma * next_value - values[i]
        gae = delta + gamma * lam * gae
        advantages[i] = gae
        next_value = values[i]
    returns = [adv + v for adv, v in zip(advantages, values)]
    return advantages, returns


def ppo_update(net, optimizer, batch, device,
               clip_ratio: float = 0.2, value_coef: float = 0.5,
               entropy_coef: float = 0.01, n_epochs: int = 4,
               minibatch: int = 256):
    """Run PPO updates on a collected batch."""
    states = torch.from_numpy(np.stack([b[0] for b in batch])).float().to(device)
    masks = torch.from_numpy(np.stack([b[1] for b in batch])).bool().to(device)
    actions = torch.tensor([b[2] for b in batch], dtype=torch.long, device=device)
    old_logprobs = torch.tensor([b[3] for b in batch], dtype=torch.float, device=device)
    advantages = torch.tensor([b[6] for b in batch], dtype=torch.float, device=device)
    returns = torch.tensor([b[7] for b in batch], dtype=torch.float, device=device)
    # Normalize advantages
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    N = len(states)
    for _ in range(n_epochs):
        perm = torch.randperm(N)
        for i in range(0, N, minibatch):
            idx = perm[i:i+minibatch]
            s = states[idx]; m = masks[idx]; a = actions[idx]
            old_lp = old_logprobs[idx]; adv = advantages[idx]; ret = returns[idx]
            logits, v = net(s)
            dist = masked_policy(logits, m)
            new_lp = dist.log_prob(a)
            entropy = dist.entropy().mean()
            ratio = (new_lp - old_lp).exp()
            clip_adv = torch.clamp(ratio, 1 - clip_ratio, 1 + clip_ratio) * adv
            policy_loss = -torch.min(ratio * adv, clip_adv).mean()
            value_loss = ((v - ret) ** 2).mean()
            loss = policy_loss + value_coef * value_loss - entropy_coef * entropy
            optimizer.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 0.5)
            optimizer.step()


def eval_vs_smart(net, device, num_games: int = 30, target: int = 100) -> float:
    """Returns win rate against Smart over num_games matches."""
    wins = 0
    for s in range(num_games):
        rng = random.Random(s * 7919 + 17)
        # Alternate sides each game for fairness
        net_player = s % 2
        g = Game(target=target, names=['Net','Smart'], rng=rng)
        obs = Observer(net_player)
        steps = 0
        while g.phase != 'gameover' and steps < 3000:
            if g.phase == 'roundover':
                g.next_round()
                obs.maybe_reset(g)
                continue
            p = g.turn
            try:
                if p == net_player:
                    obs.maybe_reset(g)
                    state = encode_state(g, p, obs)
                    mask = legal_action_mask(g, p)
                    if not mask.any(): break
                    aidx, _, _ = policy_action(net, state, mask, device, deterministic=True)
                    a = action_to_move(g, p, aidx)
                else:
                    a = bot_smart(g, p)
                    obs.record_opp_action(p, a, list(g.floor))
                apply_action(g, p, a)
            except Exception:
                break
            steps += 1
        if g.winner == net_player:
            wins += 1
    return wins / num_games


## Cell 9: Behavior cloning warm-start (~5–10 min)

Collect ~30K state/action pairs from Smart vs Smart games, then supervised-train the policy
to imitate Smart. After this, the policy plays roughly at Smart level, which is a much better
starting point for PPO than random.

In [ ]:
print('Collecting BC samples from Smart vs Smart games...')
states, actions, masks = collect_smart_samples(num_games=2000, seed=42)
print(f'Collected {len(states)} samples (state dim {states.shape[1]}).')

net = PolicyValueNet().to(device)
print(f'Network: {count_params(net):,} params')
print('Behavior cloning...')
train_bc(net, states, actions, masks, epochs=8, batch_size=512, lr=1e-3, device=device)

# Save BC checkpoint
torch.save(net.state_dict(), '/content/policy_bc.pt')
print('Saved /content/policy_bc.pt')

# Quick eval vs Smart
print('Evaluating BC policy vs Smart...')
wr = eval_vs_smart(net, device, num_games=30, target=50)
print(f'BC win rate vs Smart at target=50: {wr*100:.0f}%')


## (Optional) Resume from a saved checkpoint

If Colab disconnected mid-training, run this cell instead of Cell 9 (BC) to pick up from the
last checkpoint. Then continue with Cell 10.

In [ ]:
import os
# Try latest checkpoint first, then best
ckpt = None
for path in ['/content/policy_latest.pt', '/content/policy_best.pt', '/content/policy_bc.pt']:
    if os.path.exists(path):
        ckpt = path
        break

if ckpt is None:
    print('No checkpoint found. Run Cell 9 (BC) first.')
else:
    net = PolicyValueNet().to(device)
    net.load_state_dict(torch.load(ckpt, map_location=device))
    net.train()
    print(f'Resumed from {ckpt}')
    # Quick eval to confirm
    wr = eval_vs_smart(net, device, num_games=20, target=50)
    print(f'Current win rate vs Smart: {wr*100:.0f}%')


## Cell 10: PPO self-play with league + value pretrain + anti-collapse

Improved over the first run. Key changes:
- **Value pretrain** before PPO: critic learns Monte Carlo returns from Smart games (prevents bad-value→bad-gradient cascade).
- **LR**: 1e-4 (was 3e-4) — slower, more stable updates.
- **Clip ratio**: 0.1 (was 0.2) — smaller per-step policy changes.
- **40% Smart in league** (was 20%) — anchors policy.
- **Linear LR warmup** for first 10K games.
- **Reward shaping**: small per-step reward for hand-pt reduction (denser signal).
- **Anti-collapse**: if win rate drops 10pp below best for 3 evals, roll back to best + halve LR.

Run AFTER Cell 9 (BC). ~4-8 hours.

In [ ]:
# Hyperparams
TOTAL_GAMES = 200_000
EVAL_EVERY = 5_000
CKPT_EVERY_MIN = 30
BATCH_SIZE = 4096
LR_BASE = 1e-4              # Was 3e-4 (lowered for stability)
LR_WARMUP_GAMES = 10_000
CLIP_RATIO = 0.1            # Was 0.2 (tighter)
TARGET = 50
SMART_FRAC = 0.40           # Was 0.20

# Step 1: Pretrain value head on Smart-vs-Smart MC returns
print('Pretraining value head on Smart returns...')
pretrain_value(net, device, num_games=600, epochs=3, batch_size=256, lr=5e-4)

# Eval after value pretrain
net.eval()
wr = eval_vs_smart(net, device, num_games=30, target=TARGET)
print(f'After BC + value pretrain: win rate vs Smart = {wr*100:.0f}%')

# Step 2: PPO
net.train()
optimizer = optim.Adam(net.parameters(), lr=LR_BASE)

import copy
def snapshot():
    snap = PolicyValueNet().to(device)
    snap.load_state_dict(copy.deepcopy(net.state_dict()))
    snap.eval()
    return snap

league = [snapshot()]
best_winrate = wr  # start from post-BC+value baseline
torch.save(net.state_dict(), '/content/policy_best.pt')
last_ckpt_t = time.time()
games_played = 0
batch = []
rng = random.Random(0)
regression_count = 0  # consecutive evals below best

start_t = time.time()
print(f'Starting PPO at {time.strftime("%H:%M:%S")}')

while games_played < TOTAL_GAMES:
    # LR warmup
    if games_played < LR_WARMUP_GAMES:
        warmup_lr = LR_BASE * (0.1 + 0.9 * games_played / LR_WARMUP_GAMES)
        for g_ in optimizer.param_groups:
            g_['lr'] = warmup_lr

    opp_kind, opponent = pick_opponent(league, device,
                                       lambda g, p: bot_smart(g, p),
                                       rng, smart_frac=SMART_FRAC)
    traj, winner = play_self_game(net, opponent, device, target=TARGET,
                                  rng=random.Random(), shape_reward=True)
    games_played += 1

    advs, rets = compute_gae(traj)
    for k, t in enumerate(traj):
        batch.append(t + (advs[k], rets[k]))

    if len(batch) >= BATCH_SIZE:
        ppo_update(net, optimizer, batch, device, clip_ratio=CLIP_RATIO)
        batch = []

    if games_played % EVAL_EVERY == 0:
        net.eval()
        wr = eval_vs_smart(net, device, num_games=40, target=TARGET)
        elapsed_min = (time.time() - start_t) / 60
        cur_lr = optimizer.param_groups[0]['lr']
        print(f'[{games_played:>6}/{TOTAL_GAMES} games, {elapsed_min:.0f} min, lr={cur_lr:.1e}] win vs Smart: {wr*100:.0f}%')
        if wr > best_winrate:
            best_winrate = wr
            regression_count = 0
            torch.save(net.state_dict(), '/content/policy_best.pt')
            print(f'  ↑ new best ({wr*100:.0f}%), saved /content/policy_best.pt')
        elif wr < best_winrate - 0.10:
            regression_count += 1
            print(f'  ↓ regression ({wr*100:.0f}% vs best {best_winrate*100:.0f}%); count={regression_count}/3')
            if regression_count >= 3:
                # Roll back to best and lower LR
                net.load_state_dict(torch.load('/content/policy_best.pt'))
                new_lr = max(optimizer.param_groups[0]['lr'] * 0.5, 1e-5)
                for g_ in optimizer.param_groups:
                    g_['lr'] = new_lr
                regression_count = 0
                print(f'  ↩ rolled back to best, LR -> {new_lr:.1e}')
        else:
            regression_count = max(0, regression_count - 1)
        net.train()
        if games_played % (EVAL_EVERY * 2) == 0 and len(league) < 10:
            league.append(snapshot())

    if time.time() - last_ckpt_t > CKPT_EVERY_MIN * 60:
        torch.save(net.state_dict(), '/content/policy_latest.pt')
        last_ckpt_t = time.time()

torch.save(net.state_dict(), '/content/policy_final.pt')
print(f'Done. Best win rate vs Smart: {best_winrate*100:.0f}%')


## Cell 11: Final evaluation

Loads the best checkpoint, plays 100 matches vs Smart at target=100, reports win rate.

In [ ]:
# Load the best checkpoint
best_net = PolicyValueNet().to(device)
best_net.load_state_dict(torch.load('/content/policy_best.pt', map_location=device))
best_net.eval()
print('Loaded /content/policy_best.pt')

# Evaluate at target=50
wr50 = eval_vs_smart(best_net, device, num_games=100, target=50)
print(f'Win rate vs Smart at target=50: {wr50*100:.0f}% ({int(wr50*100)} of 100)')

# Evaluate at target=100
wr100 = eval_vs_smart(best_net, device, num_games=50, target=100)
print(f'Win rate vs Smart at target=100: {wr100*100:.0f}% ({int(wr100*50)} of 50)')


## Cell 12: Export weights for the browser

Saves the trained network's weights as a JSON file. Download it and send the path back to me;
I'll integrate it into `index.html` with a pure-JS inference function.

In [ ]:
import json
weights = {}
for name, param in best_net.state_dict().items():
    weights[name] = param.detach().cpu().numpy().tolist()

with open('/content/policy_weights.json', 'w') as f:
    json.dump({
        'state_dim': STATE_DIM,
        'action_dim': ACTION_DIM,
        'hidden_dim': 256,
        'weights': weights,
        'training_info': {
            'win_rate_vs_smart_t50': wr50,
            'win_rate_vs_smart_t100': wr100,
        }
    }, f)

import os
size_kb = os.path.getsize('/content/policy_weights.json') / 1024
print(f'Saved /content/policy_weights.json ({size_kb:.0f} KB)')

# Trigger download in Colab
from google.colab import files
files.download('/content/policy_weights.json')
